# Gradient Descent, Explained Simply
How machine learning models learn: walking downhill on the error, step by step. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. Walking downhill on a simple curve
The **loss** measures how wrong we are. Here the loss is (w - 3) squared, lowest at w = 3. Its **slope** (gradient) is 2 (w - 3). Each step: **w = w - learning rate x slope**.

### Six steps downhill
- `w = 0.0`: start with a guess
- `slope = 2 * (w - 3)`: how steep the loss curve is at w, and which way is uphill
- `w - learning_rate * slope`: take a small step the opposite way, downhill
- `the output`: w creeps towards 3, and the steps shrink as the curve flattens

In [ ]:
w = 0.0
learning_rate = 0.1
for step in range(6):
    slope = 2 * (w - 3)
    w = w - learning_rate * slope
    print(f"step {step + 1}: slope {slope:5.2f}  ->  w = {w:.3f}")

### Learning rate: too small, just right, too big
- `for lr in [...]`: four learning rates, each starting from w = 0
- `path`: remember w after every step, to draw it
- `0.02`: too small: still far from 3 after 20 steps
- `0.9`: big: jumps back and forth across 3, but settles
- `1.05`: too big: every jump overshoots more, and w runs away

In [ ]:
import matplotlib.pyplot as plt

for lr in [0.02, 0.1, 0.9, 1.05]:
    w, path = 0.0, [0.0]
    for step in range(20):
        w = w - lr * 2 * (w - 3)
        path.append(w)
    plt.plot(path, marker="o", label=f"learning rate {lr}")
    print(f"learning rate {lr}: w after 20 steps = {w:.2f}")
plt.axhline(3, color="gray", linestyle="--")
plt.xlabel("step"); plt.ylabel("w")
plt.legend(); plt.ylim(-6, 10); plt.show()

## 2. The ice cream shop
An ice cream shop records the temperature and the ice creams sold on 30 days. We fit the line **sales = w x temperature + b** by gradient descent.

### 30 days of sales
- `ice_cream.csv`: 30 days: temperature in degrees Celsius, and ice creams sold
- `.values`: plain NumPy arrays, for fast maths

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "gradient-descent/ice_cream.csv")
df = pd.read_csv(url)
x = df["temperature"].values
y = df["ice_creams"].values
print(df.shape)
df.head()

### The loss: mean squared error
- `pred = w * x + b`: the line's prediction for every day
- `np.mean((pred - y) ** 2)`: mean squared error: the average squared miss
- `three guesses`: better lines give a smaller loss

In [ ]:
import numpy as np

def loss(w, b):
    pred = w * x + b
    return np.mean((pred - y) ** 2)

for w, b in [(0, 0), (3, 0), (5, -20)]:
    print(f"w = {w}, b = {b}:  loss = {loss(w, b):,.0f}")

### Gradient descent on the raw temperatures
- `error = w * x + b - y`: how far each prediction is from the truth
- `2 * np.mean(error * x)`: the slope of the loss for w: error times input, averaged
- `2 * np.mean(error)`: the slope of the loss for b: just the average error
- `lr=0.001`: 20,000 steps and still not quite at the best line (w about 5.6, b about -32)
- `lr=0.003`: slightly bigger, and it blows up: e+90 means a number with about 90 digits

In [ ]:
def descend(x, lr, steps):
    w, b = 0.0, 0.0
    for step in range(steps):
        error = w * x + b - y
        w = w - lr * 2 * np.mean(error * x)
        b = b - lr * 2 * np.mean(error)
    return w, b

w, b = descend(x, lr=0.001, steps=20000)
print(f"lr 0.001, 20000 steps: w = {w:.2f}, b = {b:.1f}")
w, b = descend(x, lr=0.003, steps=200)
print(f"lr 0.003, 200 steps:   w = {w:.2g}, b = {b:.2g}")

## 3. A simple fix: centre the input
Temperatures are around 25, so the w direction is far steeper than the b direction. Measuring temperature as **degrees above or below average** balances them.

### Centred temperatures
- `x - x.mean()`: degrees above (+) or below (-) the average day
- `history`: the loss after every step, to plot
- `step % 50 == 0`: print every 50th step
- `the output`: the loss drops fast, then levels off: we've reached the bottom of the valley

In [ ]:
x_c = x - x.mean()
w, b = 0.0, 0.0
lr = 0.01
history = []
for step in range(301):
    error = w * x_c + b - y
    w = w - lr * 2 * np.mean(error * x_c)
    b = b - lr * 2 * np.mean(error)
    history.append(np.mean(error ** 2))
    if step % 50 == 0:
        print(f"step {step:3}: w={w:.2f} b={b:.1f} loss={history[-1]:,.0f}")

### Back to normal temperatures, and a check
- `b - w * x.mean()`: convert the intercept back to ordinary temperatures
- `LinearRegression`: scikit-learn's exact answer, for comparison
- `the output`: the same line, found by walking downhill

In [ ]:
b_raw = b - w * x.mean()
print(f"gradient descent: w = {w:.2f}, b = {b_raw:.1f}")

from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(df[["temperature"]], y)
print(f"scikit-learn:     w = {m.coef_[0]:.2f}, b = {m.intercept_:.1f}")

### The loss, step by step
- `plt.plot(history)`: the loss after every step
- `the shape`: steep at first, then flat: the typical learning curve

In [ ]:
plt.plot(history)
plt.xlabel("step")
plt.ylabel("loss (mean squared error)")
plt.title("Walking downhill")
plt.show()

### The line getting better
- `plt.scatter`: the 30 days, in grey
- `step in [0, 5, 20, 300]`: draw the current line at four moments
- `the picture`: flat at first, then tilting and rising until it fits the dots

In [ ]:
plt.scatter(x, y, color="gray")
w, b = 0.0, 0.0
for step in range(301):
    if step in [0, 5, 20, 300]:
        plt.plot(x, w * x_c + b, label=f"step {step}")
    error = w * x_c + b - y
    w, b = (w - lr * 2 * np.mean(error * x_c),
            b - lr * 2 * np.mean(error))
plt.xlabel("temperature (C)"); plt.ylabel("ice creams sold")
plt.legend(); plt.show()

## 4. Mini-batches
Real datasets can have millions of rows. **Mini-batch gradient descent** estimates the slope from a small random sample each step.

### Mini-batch gradient descent
- `rng.choice(..., size=10)`: pick 10 random days for this step
- `x_c[idx], y[idx]`: only those 10 days
- `0.003`: a smaller learning rate, because each step's slope is a noisy estimate
- `the output`: close to the exact answer (5.64, -32.2), using a third of the data per step; it wobbles a little around the bottom

In [ ]:
rng = np.random.default_rng(0)
w, b = 0.0, 0.0
for step in range(3000):
    idx = rng.choice(len(x), size=10, replace=False)
    error = w * x_c[idx] + b - y[idx]
    w = w - 0.003 * 2 * np.mean(error * x_c[idx])
    b = b - 0.003 * 2 * np.mean(error)
print(f"mini-batch: w = {w:.2f}, b = {b - w * x.mean():.1f}")

## 5. Use the line
Predict a hot day.

### Predict 30 degrees
- `w_final * 30 + b_final`: plug 30 into the line
- `the answer`: how many to prepare for a hot day

In [ ]:
w_final, b_final = m.coef_[0], m.intercept_
print(round(w_final * 30 + b_final), "ice creams on a 30 degree day")

## Try it yourself
1. In the first cell, start from w = 10 instead of 0. Does it still reach 3?
2. Try a learning rate of 0.5 on the ice cream data. What happens?
3. Change the mini-batch size from 10 to 3. Is the path smoother or noisier?